In [ ]:
# allows update of external libraries without need to reload package
%load_ext autoreload
%autoreload 2

# Data product demonstration: Terrain complexity assessment and representativity

## Summary

A core decision in wind reports is whether reference measurements of wind are representative for any site of a planned turbine. Subsequently we demonstrate the application and the artifacts of some assessment.

Demonstations include:

- The preparation of data, i.e. DEMs and locations within the map's boundaries.
- The assessment itself including configuration.
- Inspecting results and details of the assessment.
- Writing artifacts including information about the input data and the processing.

## Results

### Key results

- Prerequisites:
  - A DEM equipped w/ a CRS as a dataarray, one or two geodataframes holding the locations in a projected metric CRS.
  - Metadata about the DEM.
- Run:
  - `RIXAnalyzer`: Manages computations and gathering metadata.
- Artifacts:
  - `Writer`: Generates the artifacts.

## Features

1. All definitions are fetched from two central objects:
   - `PRODUCT_DEFINITION_TRIX`: Holds parameters, metadata, table schemes.
   - `ColumnKeys`: Holds all keys that appear somewhere. # TODO: Separate internal and external keys.
1. Analysis generates several intermediate and detailed results for validation.
1. Writer generates artifacts for further processing: Manifest containing metadata, tables and a geopackage.

## Workflow

1. Set up data:
   - DEM as a `xarray.DataArray` with its own CRS. If coordinates differ from `("y", "x")`, adapt in `ColumnKeys`.
   - Locations within the bounds of the map and a sufficiently large buffer as `gpd.GeoDataFrame`:
     - Coordinates of the sites to assess, the planned WTGs.
     - Coordinates of the reference sites, the wind data base locations.
     - Provide coordinates in a projected metric CRS.
   - Metadata of the DEM.
1. Run the analysis.
1. Inspect results.
1. Save artifacts.

## Examples

Examples are prepared using synthetic data: A planar map that slopes down eastwards, and is equipped with WGS84, and reprojected to UTM 33N (EPSG:32633). Any site is provided in the latter CRS, any for the non-example transformed to EPSG:4326.

1. Locations in UTM, map in GCS.
1. Locations in UTM, map in UTM.
1. Locations in GCS, map in GCS: This is a non-example.
1. Locations w/o CRS, map w/o CRS, include:
   - Explicit computation.
   - Behaviour at the boundary.

## Imports and Setup

In [ ]:
import copy

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import shapely
import shapely.plotting
import yaml

import phoibe.geography.crs
import phoibe.synthetic_data.fields
import phoibe.synthetic_data.sites
from phoibe.geography.complexity.rix import analyzer, config, writer

In [ ]:
def print_dict(dictionary):
    """Print a dictionary nicely."""
    print(yaml.dump(dictionary, sort_keys=False))


def plot_rose(rix_directional: pd.Series, label: str = None, ax=None, figure=None):
    """Polar plot angular data. Pass angles as index in [°]."""
    angles_radians = np.deg2rad(np.append(rix_directional.index, rix_directional.index[0]))
    values = rix_directional.values
    values = np.append(values, values[0])

    if ax is None:
        figure, ax = plt.subplots(subplot_kw={"projection": "polar"})

    ax.plot(angles_radians, values, label=label, linewidth=0.7)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_ylabel("RIX")

    return figure, ax

## Create scenario

## Create map and equip w/ CRS

We assess sites on a planar map that that ascends from west to east. This map gets equipped w/ WGS84 coordinates. As a projected metric CRS, we choose an UTM 33N CRS.

Although not strictly necessary for the assessment, we provide the map in a projected CRS. `meta_reproject` provides context about the reprojection:

```yaml
crs_from: EPSG:4326
bounds_src:
  left: 12.699378109452736
  bottom: 52.100373134328365
  right: 13.199378109452736
  top: 52.400373134328355
range_src:
  width: 402
  height: 402
range_dst:
  width: 439
  height: 430
transform: !!python/object/new:affine.Affine
- 80.0
- 0.0
- 342424.89372122474
- 0.0
- -80.0
- 5808060.59357695
nodata_src: 0
nodata_dst: 10692
```

In [ ]:
# generate
NX, NY = 201, 201
DX, DY = 10, 10
SLOPE_X, SLOPE_Y = 1.0, 0.0
BOUNDS = (12.7, 52.1, 13.2, 52.4)
CRS_TO = "32633"
BOUNDS_UTM = (342425, 5773661, 377545, 5808061)

plane_vanilla = phoibe.synthetic_data.fields.make_planar_field(
    nx=NX, ny=NY, dx=DX, dy=DY, slope_x=SLOPE_X, slope_y=SLOPE_Y
)
meta_plane_vanilla: dict = {
    "name": "Planar field",
    "source": "Synthetic data",
    "description": "2D field w/o any CRS information.",
}

plane_gcs = phoibe.synthetic_data.fields.make_field_rio(da=plane_vanilla, bounds=BOUNDS, crs="4326", dtype="float")
meta_plane_gcs: dict = {
    "name": "Planar field",
    "source": "Synthetic data",
    "description": "2D field equipped w/ CRS EPSG:4326.",
}

plane_utm, meta_reproject = phoibe.geography.crs.reproject.reproject_rasterdata(
    da=plane_gcs, crs_to=CRS_TO, resolution=80, resampling=rasterio.warp.Resampling.bilinear
)
meta_plane_utm: dict = {
    "name": "Planar field",
    "source": "Synthetic data",
    "description": f"2D field equipped w/ CRS EPSG:{CRS_TO}.",
}

print_dict(meta_reproject)

### Create sites

Create some sites:

- `sites_utm`: These 3 sites are going to be assessed.
- `references_utm`: These 2 sites host the reference measurements.

In [ ]:
# generate
SITE_IDS = [f"WTG {k}" for k in range(3)]
SITES = [shapely.Point(363000, 5794000), shapely.Point(363000, 5797000), shapely.Point(363000, 5787000)]

REFERENCE_IDS = [f"M {k}" for k in range(2)]
REFERENCES = [shapely.Point(362900, 5794000), shapely.Point(363050, 5787000)]

sites_utm = gpd.GeoDataFrame(data={"location_id": SITE_IDS, "geometry": SITES}, geometry="geometry", crs=CRS_TO)
references_utm = gpd.GeoDataFrame(
    data={"location_id": REFERENCE_IDS, "geometry": REFERENCES}, geometry="geometry", crs=CRS_TO
)

figure, axes = phoibe.geography.plot.plot_raster(da=plane_utm)

sites_utm.plot(color="r", ax=axes)
for _k, row in sites_utm.iterrows():
    axes.annotate(xy=(row.geometry.x, row.geometry.y), text=row["location_id"], fontsize=17)

references_utm.plot(color="b", ax=axes)
for _k, row in references_utm.iterrows():
    axes.annotate(xy=(row.geometry.x, row.geometry.y), text=row["location_id"], fontsize=17)

## Analyse complexity

### In UTM w/ map in GCS

The analyzer gets instantiated with the follwoing parameters. Update parameters as desired. Concerning the parameters:

- `n_angles`, `R_km` and `slope_critical` are the locked parameters from TR6.
- `dr_km` is the stepsize for a grid along each ray for sampling from DEM raster data.
- `interpolation_method` determines how the off-grid elevations are determined from the raster data.
  - Note: The ray grid points are determined in the sites' given CRS, and then transformed into the DEM native CRS to sample the elevations. The interpolation method can be `linear` or `nearest`. Be aware of their respective side-effects.
- `crs` is currently unused.

```yaml
name: T-RIX assessment
version: '1.0'
description: TRIX, TR6 Rev.12
parameters:
  n_angles: 72
  R_km: 3.5
  dr_km: 0.05
  slope_critical: 0.033
  crs: null
sampler:
  interpolation_method: linear
```

The analysis run consumes the DEM, `sites_utm` and, optionally, `references_utm`. If the latter is missing, only RIX and steep segments are determined.

In [ ]:
# show
print_dict(config.ANALYZER_DEFAULTS)

In [ ]:
# compute
cfg = copy.deepcopy(config.ANALYZER_DEFAULTS)
cfg["parameters"].update({"crs": plane_gcs.rio.crs.to_string()})
rix_analyzer = analyzer.TRIXAnalyzer(config=cfg)

rix_results = rix_analyzer.run(
    dem=plane_utm, locations_site=sites_utm, locations_reference=references_utm, dem_metadata=meta_plane_utm
)

### Results

The analysis results are structured as follows:

- `locations_site`, `locations_reference`: The input location dataframes.
- `summary`: Table of ruggedness analysis, includes elevations (as mean of ray origin elevations), and ruggedness summary statistics.
- `ruggedness_roses`: For each site, the mapping of the direction to the directional ruggedness.
- `steep_segments`: Determined steep segments for each site.
- `trix_table`: Table containing the representativity decision, distance, T-RIX and distance measures A and B for each pair of site and reference. Empty if no reference provided.
- `meta`: Metadata of the analysis, including parameters, and data context.

In [ ]:
# show
print(sorted([item for item in rix_results.__dir__() if not item.startswith("_")]))

In [ ]:
# show
figure, axes = phoibe.geography.plot.plot_raster(da=plane_utm)
shapely.plotting.plot_polygon(sites_utm.loc[0].geometry.buffer(3500), ax=axes)

rix_results.steep_segments.plot(color="r", ax=axes, linewidth=0.7)

sites_utm.plot(color="b", ax=axes)
for _k, row in sites_utm.iterrows():
    axes.annotate(xy=(row.geometry.x, row.geometry.y), text=row["location_id"], fontsize=17)

Contents of the tables:

- `trix_table`: 
  - transferability, 2-yes, 1-with extra uncertainty, 0-no.
  - distance between both sites [km].
  - trix index between both sites, a weighted average of the absolute differences in RIX and elevations.
  - A and B are the threshold distances for the transferability.
- `summary`:
  - elevation determined from the elevation of the ray origins. As they should agree, the standard deviation should be 0 (deviations indicate unexpected behaviour).
  - rix and its summary statistics over the given rays.
  - parameters: number of rays and the critical slope.

Metadata:

- `meta`: Information about the run.
- `parameters`: Parameters used during the assessment.
- `spatial_context`: Data source information including:
  - `dem`: Model, related CRS, extent and resolution.
  - `ray`: CRS used for constructing the rays and counts of any NaN seen during any sampling of the sites.
  - `internal_alignment`: Information about coordinate transformations performed.

```yaml
meta:
  name: T-RIX assessment
  version: '1.0'
  description: TRIX, TR6 Rev.12
  created_at: 2026-07-07 09:29:23 UTC
parameters:
  ray:
    n_angles: 72
    R_km: 3.5
    dr_km: 0.05
  slope:
    slope_critical: 0.033
  sampler:
    interpolation_method: linear
spatial_context:
  source_dem:
    name: Planar field
    source: Synthetic data
    description: 2D field equipped w/ CRS EPSG:32633.
    crs:
    - EPSG:4326
    extent:
    - west: 12.699378109452736
      south: 52.100373134328365
      east: 13.199378109452736
      north: 52.400373134328355
    resolution:
    - dx: 0.0012437810945273632
      dy: -0.0007462686567164109
  source_ray:
    crs:
    - EPSG:32633
    nan_count: 0
  internal_alignment:
    messages:
    - Transformed ray CRS EPSG:32633 to DEM CRS EPSG:4326 for sampling.
```

The rix rose plot displays for each ray the corresponding ruggedness. For a planar elevation map, expect a quite simple plot.

In [ ]:
# show
rix_results.trix_table

In [ ]:
# show
rix_results.summary

In [ ]:
# show
print_dict(rix_results.meta)

In [ ]:
rix_results.steep_segments.loc[rix_results.steep_segments.theta == 90.0]  # .geometry[331].xy
figure, ax = plt.subplots(subplot_kw={"projection": "polar"})
for key, row in rix_results.ruggedness_roses.iterrows():
    plot_rose(row, label=str(key), ax=ax, figure=figure)

ax.legend()
ax.set_title("RIX roses")

#### Writer

The writer writes the artifacts depending on the profile and the assessment:

- SUMMARY: Metadata as a human-readable file, the RIX table and, if reference locations were assessed, the T-RIX table.
- FULL: The files in SUMMARY plus a geopackage w/ several layers: Provided locations, steep segments of the assessed sites, and the T-RIX table.

In [ ]:
# write
writer.RIXWriter(
    result=rix_results,
    locations_site=sites_utm,
    locations_reference=references_utm,
    profile=writer.WriterProfile.SUMMARY,
).write(directory="02-example-utm-gcs", project_name="WP Planar I")

### In UTM w/ map in UTM

In this example, the elevation map and sites' coordinates are given in the same CRS. 

Differences in the results are marginal.

In [ ]:
cfg = copy.deepcopy(config.ANALYZER_DEFAULTS)
cfg["parameters"].update({"crs": plane_utm.rio.crs.to_string()})
rix_analyzer = analyzer.TRIXAnalyzer(config=cfg)

rix_results = rix_analyzer.run(
    dem=plane_utm, locations_site=sites_utm, locations_reference=references_utm, dem_metadata=meta_plane_utm
)

In [ ]:
figure, axes = phoibe.geography.plot.plot_raster(da=plane_utm)
rix_results.steep_segments.plot(color="r", ax=axes, linewidth=0.7)

sites_utm.plot(color="b", ax=axes)
for _k, row in sites_utm.iterrows():
    axes.annotate(xy=(row.geometry.x, row.geometry.y), text=row["location_id"], fontsize=17)

In [ ]:
rix_results.summary

In [ ]:
# write
writer.RIXWriter(
    result=rix_results,
    locations_site=sites_utm,
    locations_reference=references_utm,
    profile=writer.WriterProfile.FULL,
).write(directory="02-example-utm-utm", project_name="WP Planar I")

### In GCS w/ map in GCS

No reasonable use case, but design allows this. Expect directions and distances to be distorted, and parameters misinterpreted.

Distance computation breaks here:
- Units:
  - Parameter's units are chosen to comply w/ the map's underlying CRS units, and these are metric.
  - Parameters are accepted in [km], and converted internally to [m] to align w/ the CRS units.
  - Output will be re-converted to [km].
  - As EPSG:4326's unit is [°], parameters are interpreted formally as "[k°]". Choose sufficiently small parameters to not run out of the map's bounds, i.e. add some `e-3` factor.
  - The slope turns then into [m/°].
- Metrics: 
  - The natural metric in this context is Euclidean. The distance measured is in [°] assuming this grid to be recangular.

The assessment recognizes the unintended use and leaves a note at `spatial_context.internal_alignment.messages`.

```yaml
meta:
  name: T-RIX assessment
  version: '1.0'
  description: TRIX, TR6 Rev.12
  created_at: 2026-07-07 10:08:36 UTC
parameters:
  ray:
    n_angles: 72
    R_km: 5.0e-05
    dr_km: 5.0e-07
  slope:
    slope_critical: 3000.0
  sampler:
    interpolation_method: linear
spatial_context:
  source_dem:
    name: Planar field
    source: Synthetic data
    description: 2D field equipped w/ CRS EPSG:4326.
    crs:
    - EPSG:4326
    extent:
    - west: 12.699378109452736
      south: 52.100373134328365
      east: 13.199378109452736
      north: 52.400373134328355
    resolution:
    - dx: 0.0012437810945273632
      dy: -0.0007462686567164109
  source_ray:
    crs:
    - EPSG:4326
    nan_count: 0
  internal_alignment:
    messages:
    - All coordinates are in the same CRS EPSG:4326. No guarantee unless sites are
      presented in a metric CRS.
```

In [ ]:
# generate
sites_gcs = sites_utm.to_crs(crs=4326)
references_gcs = references_utm.to_crs(crs=4326)

In [ ]:
# compute
cfg = copy.deepcopy(config.ANALYZER_DEFAULTS)
R_KDEG = 5e-5
DR_KDEG = 5e-7

cfg["parameters"].update(
    {"crs": plane_gcs.rio.crs.to_string(), "R_km": R_KDEG, "dr_km": DR_KDEG, "slope_critical": 3e3}
)
rix_analyzer = analyzer.TRIXAnalyzer(config=cfg)

rix_results = rix_analyzer.run(
    dem=plane_gcs,
    locations_site=sites_gcs,
    locations_reference=references_gcs,
    dem_metadata=meta_plane_gcs,
)

In [ ]:
# show
figure, axes = phoibe.geography.plot.plot_raster(da=plane_gcs)

rix_results.steep_segments.plot(color="r", ax=axes, linewidth=0.7)
sites_gcs.plot(color="b", ax=axes)
for _k, row in sites_gcs.iterrows():
    shapely.plotting.plot_polygon(row.geometry.buffer(R_KDEG * 1000), ax=axes)
    axes.annotate(xy=(row.geometry.x, row.geometry.y), text=row["location_id"], fontsize=17)

In [ ]:
# show
rix_results.summary

In [ ]:
# write
writer.RIXWriter(
    result=rix_results,
    locations_site=sites_utm,
    locations_reference=references_utm,
    profile=writer.WriterProfile.FULL,
).write(directory="02-example-gcs-gcs", project_name="WP Planar I")

### No CRS

W/o any CRS information, the assessment can still be carried out w/o any conversion. The user needs to ensure that coordinates align.

```yaml
meta:
  name: T-RIX assessment
  version: '1.0'
  description: TRIX, TR6 Rev.12
  created_at: 2026-07-07 10:19:07 UTC
parameters:
  ray:
    n_angles: 72
    R_km: 0.5
    dr_km: 0.01
  slope:
    slope_critical: 0.9
  sampler:
    interpolation_method: linear
spatial_context:
  source_dem:
    name: Planar field
    source: Synthetic data
    description: 2D field w/o any CRS information.
    crs:
    - null
    extent:
    - west: -2015.0
      south: -2015.0
      east: 2005.0
      north: 2005.0
    resolution:
    - dx: 10.0
      dy: 10.0
  source_ray:
    crs:
    - null
    nan_count: 0
  internal_alignment:
    messages:
    - 'Assume all coordinates are in the same CRS. Ray-CRS None. DEM-CRS None. '
```

Verify that in x-direction, the slope of the generated field actually is 1. As there are no distortions or rotations due to the grid, computations can be carried out explicitly:

- Slopes of rays are explixit from their angle w/ the axes.
- Slopes are constant along a ray, hence either a ray is completely rugged, or not at all.
- RIX=0.3055555555555556.



In [ ]:
# compute
print(plane_vanilla.differentiate("x"))


def expected_rix(slope_x, slope_y, slope_critical, n_angles):
    angles = np.linspace(start=0, stop=360, num=n_angles, endpoint=False)
    slope_rays = slope_x * np.sin(np.deg2rad(angles)) + slope_y * np.cos(np.deg2rad(angles))
    return float(np.mean(np.abs(slope_rays) > slope_critical))


expected_rix(slope_x=SLOPE_X, slope_y=SLOPE_Y, slope_critical=0.9, n_angles=72)

In [ ]:
# generate
bounds = (
    float(plane_vanilla.x.min()),
    float(plane_vanilla.y.min()),
    float(plane_vanilla.x.max()),
    float(plane_vanilla.y.max()),
)
sites_vanilla = (
    phoibe.synthetic_data.sites.make_sites(sites=len(SITE_IDS), bounds=bounds, buffer=0, seed=23, crs=None)
    .rename(columns={"name": "location_id"})
    .astype({"location_id": str})
)
sites_vanilla.loc[:, "location_id"] = SITE_IDS

references_vanilla = (
    phoibe.synthetic_data.sites.make_sites(sites=len(REFERENCE_IDS), bounds=bounds, buffer=200, seed=42, crs=None)
    .rename(columns={"name": "location_id"})
    .astype({"location_id": str})
)
references_vanilla.loc[:, "location_id"] = REFERENCE_IDS

In this setup, a couple of rays hit the boundary. The logger warns about every single ray that saw any NaN and how many of them. However, these counts are aggregated and part of the results.

In case of occurance, clarify the reason and mitgate. If they appear:

- At the boundary: The extent is too small. Choose a larger part of the map.
- In the interior: The input map has holes. As this is a data issue, a resolution is out of the scope of the assessment itself.

In [ ]:
# compute
cfg = config.ANALYZER_DEFAULTS.copy()
cfg["parameters"].update({"crs": None, "slope_critical": 0.9, "R_km": 0.5, "dr_km": 0.01, "n_angles": 72})
rix_analyzer = analyzer.TRIXAnalyzer(config=cfg)

rix_results = rix_analyzer.run(
    dem=plane_vanilla,
    locations_site=sites_vanilla,
    locations_reference=references_vanilla,
    dem_metadata=meta_plane_vanilla,
)

In [ ]:
# show
rix_results.summary

In [ ]:
# show
figure, axes = phoibe.geography.plot.plot_raster(da=plane_vanilla)
rix_results.steep_segments.plot(color="r", ax=axes, linewidth=0.7)

sites_vanilla.plot(color="b", ax=axes)
for _k, row in sites_vanilla.iterrows():
    axes.annotate(xy=(row.geometry.x, row.geometry.y), text=row["location_id"], fontsize=17)

references_vanilla.plot(color="b", ax=axes)
for _k, row in references_vanilla.iterrows():
    axes.annotate(xy=(row.geometry.x, row.geometry.y), text=row["location_id"], fontsize=17)

In [ ]:
# show
print_dict(rix_results.meta)

In [ ]:
# write
writer.RIXWriter(
    result=rix_results,
    locations_site=sites_utm,
    locations_reference=references_utm,
    profile=writer.WriterProfile.FULL,
).write(directory="02-example-none-none", project_name="WP Planar I")

## Read geopackage

In [ ]:
DIRECTORY = "02-example-utm-utm"
display(gpd.list_layers(DIRECTORY + "/rix_details.gpkg"))

In [ ]:
gpd.GeoDataFrame.from_file(DIRECTORY + "/rix_details.gpkg", layer="ruggedness").groupby(
    ["location_id", "theta"]
).count()